In [8]:
from splinter import Browser
from bs4 import BeautifulSoup as soup  
import time
import json
import random
import os
import shutil

In [14]:
browser = Browser('chrome')
city = "Nagpur"
target_cars = 100
cars_collected = 0
total_pages = 200

## Functions

1. Scraper Function

In [23]:
def collect_car_links(city, total_pages, target_cars):
    cars_collected = 0
    output_file = f"car_links_{city}.txt"
 
    existing_links = set()
    if os.path.exists(output_file):
        with open(output_file, 'r') as f:
            existing_links = set(line.strip() for line in f if line.strip())
        print(f" Loaded {len(existing_links)} existing links.")

    with open(output_file, "a") as f:
        for page_num in range(1, total_pages + 1):

            if page_num == 1:
                url = f"https://www.cardekho.com/used-cars+in+{city}"
            else:
                url = f"https://www.cardekho.com/used-cars+in+{city}/page-{page_num}"

            browser.visit(url)
            time.sleep(2)
            browser.execute_script("window.scrollTo(0, 1000);")
            time.sleep(1)

            current_soup = soup(browser.html, 'html.parser')

            page_links_found = 0

            for link in current_soup.find_all('a', href=True):
                href = link['href']

                if 'used-car-details' in href:
                    full_url = f"https://www.cardekho.com{href}" if href.startswith('/') else href
 
                    if full_url not in existing_links:
                        f.write(full_url + "\n")
                        existing_links.add(full_url)  
                        cars_collected += 1
                        page_links_found += 1

            print(f"Page {page_num}: Saved {page_links_found} new links. Total: {cars_collected}")

            if cars_collected >= target_cars:
                print("Target reached!")
                break

    print(f"All links saved in {output_file}. Total unique: {len(existing_links)}")

2. Extraction Function

In [ ]:
def scrape_car_links(city, links_filename="car_links.txt"):
    city = city.title()
    def get_browser():
        return Browser('chrome')

    # 1. Load links
    with open(links_filename, "r") as f:
        all_links = [line.strip() for line in f.readlines()]

    output_file = f"car_dataset_{city}.json"
    browser = get_browser()

    for index, link in enumerate(all_links):
        try:
            print(f"Scraping {index+1}/{len(all_links)}: {link}")

            # Visit page
            browser.visit(link)

            # Human delay + scroll
            time.sleep(random.uniform(1, 2))
            browser.execute_script("window.scrollTo(0, 600);")
            time.sleep(0.5)

            # Expand specifications
            try:
                view_all_spec_btn = browser.find_by_text('View all Specifications')
                if view_all_spec_btn:
                    browser.execute_script(
                        "arguments[0].click();",
                        view_all_spec_btn.first._element
                    )
                    print("Expanded specifications.")
                    time.sleep(0.6)
            except Exception:
                pass

            # Parse page
            page_soup = soup(browser.html, 'html.parser')

            car_data = {"url": link}

            # -------------------------
            # CAR NAME EXTRACTION
            # -------------------------
            name_tag = page_soup.find('div', class_='vehicleName')
            h1 = name_tag.find('h1') if (name_tag and name_tag.find('h1')) else page_soup.find('h1')

            if h1:
                parts = h1.get_text(separator="|", strip=True).split("|")
                car_data["car_name"] = parts[1].strip() if len(parts) >= 2 else parts[0].strip()

            # -------------------------
            # PRICE EXTRACTION
            # -------------------------
            price_div = page_soup.find('div', class_='vehiclePrice')
            if price_div:
                price_span = price_div.find('span')
                if price_span:
                    car_data["Price"] = price_span.get_text(strip=True)

            # -------------------------
            # SPECIFICATIONS EXTRACTION
            # -------------------------
            spec_items = page_soup.find_all('li', class_='gsc_col-xs-12')

            for item in spec_items:
                label_tag = item.find('div', class_='label')
                value_tag = item.find('span', class_='value-text')

                if label_tag and value_tag:
                    label = label_tag.get_text(strip=True)
                    value = value_tag.get_text(strip=True)
                    car_data[label] = value

            # Save data
            if len(car_data) > 1:
                with open(output_file, "a") as out:
                    out.write(json.dumps(car_data) + "\n")

                print(f"Saved: {car_data.get('Price','N/A')} and {len(car_data)-2} specs.")
            else:
                print(f"No data found for: {link}")

        except Exception as e:
            print(f"Serious error at {link}: {e}")

            browser.quit()
            browser = get_browser()
            time.sleep(1)
            continue

    browser.quit()

3. Helper

In [25]:
def is_city_processed(city, processed_file='processed.txt'):
    if not os.path.exists(processed_file):
        return False
    with open(processed_file, 'r') as f:
        return city in f.read().splitlines()

def mark_city_processed(city, processed_file='processed.txt'):
    with open(processed_file, 'a') as f:
        f.write(city + '\n')  

In [27]:
def process_city(city, total_pages, target_cars):
    city = city.title()

    # ─── Step 1: Collect Links ───────────────────────
    if is_city_processed(city):
        print(f"Links already collected for {city} — skipping!")
    else:
        collect_car_links(city, total_pages, target_cars)
        mark_city_processed(city)

        with open(f'car_links_{city}.txt', 'r') as f:
            lines = [line.strip() for line in f if line.strip()]
        total      = len(lines)
        unique     = len(set(lines))
        duplicates = total - unique
        print(f"Total: {total} | Unique: {unique} | Duplicates: {duplicates}")
        print(f"{city} links collected!")

    # ─── Step 2: Scrape Car Data ─────────────────────
    txt_file = f'car_links_{city}.txt'
    json_file = f'car_dataset_{city.lower()}.json'

    if not os.path.exists(txt_file):
        print(f"No links file found for {city} — cannot scrape!")
        return

    if os.path.exists(json_file):
        print(f"Already scraped: {city} — skipping!")
    else:
        print(f"Starting scraping for {city}...")
        scrape_car_links(city, txt_file)
        print(f"{city} scraping done!")

3. Organize

In [ ]:
def organize_city_files(city, base_dir='.'):
    city = city.title()
    json_file = f'car_dataset_{city}.json'
    txt_file  = f'car_links_{city}.txt'

    json_exists = os.path.exists(json_file)
    txt_exists  = os.path.exists(txt_file)

    if not json_exists and not txt_exists:
        print(f"⚠️  No files found for {city} — skipping!")
        return

    city_parent = os.path.join(base_dir, 'CITY')
    if not os.path.exists(city_parent):
        print(f"❌ Parent folder not found: {city_parent}")
        return

    # create the final city folder
    city_folder = os.path.join(city_parent, city)
    if not os.path.exists(city_folder):
        os.mkdir(city_folder)
        print(f"📁 Created folder: {city_folder}")
    else:
        print(f"📁 Folder already exists: {city_folder}")

    if json_exists:
        shutil.move(json_file, os.path.join(city_folder, json_file))
        print(f"✅ Moved: {json_file} → {city_folder}")
    else:
        print(f"⚠️  JSON not found for {city} — skipping!")

    if txt_exists:
        shutil.move(txt_file, os.path.join(city_folder, txt_file))
        print(f"✅ Moved: {txt_file} → {city_folder}")
    else:
        print(f"⚠️  TXT not found for {city} — skipping!")

    print(f"🎉 {city} files organized!\n")

---

## Usage

In [ ]:
process_city(city, total_pages, target_cars)

In [30]:
organize_city_files(city)

📁 Created folder: .\CITY\Nagpur
✅ Moved: car_dataset_nagpur.json → .\CITY\Nagpur
✅ Moved: car_links_Nagpur.txt → .\CITY\Nagpur
🎉 Nagpur files organized!



In [ ]:
# Make one processed.txt file inside the web_scraping folder
# Add the processed city to it
# publish the web_scraping part separately 
